Copyright (c) 2025 Mitsuru Ohno  
Use of this source code is governed by a BSD-3-style  
license that can be found in the LICENSE file.  

## 当ノートブックのワークフロー  
複数のデータフレームを横断的に処理する事例。3成分で相互平衡になっている。  
1. 未知の速度定数を含む、csvに書き込んだ反応式を読み込む。  
2. 化学種の濃度の経時変化の実験データを読み込む。初期濃度を変化させた複数の条件での実験結果を用い、全体を最適化できる速度定数を求めることを目標とする。  
3. 化学種ごとの微分型の反応速度式を作成する。速度定数が未知の素反応にはシンボリックな変数が割り振られる。  
4. 作成した微分方程式を、数値解析可能な形式にする。  
5. **P0OptFit**（Optuna）で初期パラメータ p0 を探索し、最適な p0 で **ExpDataFitSci** により速度定数をフィッティングする。本ノートブックでは use_log_fit=True を用いる。  
6. 経時変化を図示する。  

残念ながら、df2の結果を再現できていない。複数のデータフレームを横断的に処理する実行例として掲載する。  

## 反応式を記載したcsvファイルを指定する  

In [ ]:
file_path_rxn = './sample_data/ref5/sample_rxn_ref5.csv'  # CSVファイルのパスを指定

In [ ]:
# 確認後削除するセル
import sys
from pathlib import Path

# ノートブックが examples/ にあるとき、src/rxnfit を import できるようにする
_examples_dir = Path().resolve()
_src_dir = _examples_dir.parent / "src"
if _src_dir.exists():
    sys.path.insert(0, str(_src_dir))

## 反応速度式をscipy.integrate.solve_ivpで処理できる連立微分方程式にする  

In [ ]:
from dataclasses import dataclass, field
import inspect
import time #15番目のセル、Optunaで初期値最適化後データにフィットさせる処理の計時  
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from sympy import Symbol
from sympy.core.symbol import Symbol as SympySymbol

from rxnfit.build_ode import RxnODEbuild, create_system_rhs
from rxnfit.expdata_reader import expdata_read, get_y0_from_expdata
from rxnfit.expdata_fit_sci import ExpDataFitSci
from rxnfit.p0_opt_fit import P0OptFit
from rxnfit.solv_ode import SolverConfig, RxnODEsolver

# 反応速度式の作成

In [ ]:
builded_rxnode = RxnODEbuild(file_path_rxn)

In [ ]:
builded_rxnode.get_ode_info(debug_info=True)

In [ ]:
# 作成した微分方程式
builded_rxnode.get_ode_system()[0]

In [ ]:
# 速度定数の確認
print(builded_rxnode.rate_consts_dict)

check_type = [v for v in builded_rxnode.rate_consts_dict.values()]
[type(e) for e in check_type]

## 経時変化の実験データを読み込み　　
### データフレーム化  

In [ ]:
df1 = pd.read_csv(f'./sample_data/ref5/ref5_df1.csv')
df2 = pd.read_csv(f'./sample_data/ref5/ref5_df2.csv')
df3 = pd.read_csv(f'./sample_data/ref5/ref5_df3.csv')
df4 = pd.read_csv(f'./sample_data/ref5/ref5_df4.csv')

expdata_read([df1, df2, df3, df4])  # 複数データフレームのリストを渡すとまとめて読み込み可能

### フィッティング  
P0OptFit に RxnODEbuild のインスタンスと df_list を渡し、Optuna で初期値 p0 を最適化したうえで ExpDataFitSci によりフィッティングする。  


In [ ]:
t_cpu_start = time.process_time() # CPU時間計測

# P0OptFit: RxnODEbuild インスタンスと df_list を渡す（他は最小限）
opt = P0OptFit(builded_rxnode, [df1, df3, df4], use_log_fit=True)
result_dict, rss = opt.optimize(n_trials=30)

# プロット用に最適 p0 で ExpDataFitSci を1回実行し、config を取得
keys = builded_rxnode.get_symbolic_rate_const_keys()
p0_best = [result_dict[k][0] for k in keys]
t_range = (0, max(float(df.iloc[:, 0].max()) for df in [df1, df2, df3, df4]))
fit_sci = ExpDataFitSci(builded_rxnode, [df1, df3, df4], t_range)
fit_sci.run_fit(p0_best, use_log_fit=True, lower_bound=1e-10)
t_cpu_end = time.process_time()
elapsed = t_cpu_end - t_cpu_start
h, r = divmod(int(elapsed), 3600)
m, s = divmod(r, 60)
print(f"CPU time: {h:02d}:{m:02d}:{s:02d}")

In [ ]:
# フィッティング結果で builded_rxnode を更新し、SolverConfig を取得
# fit_sci.get_fitted_rate_const_dict() と get_solver_config_args() を使用
builded_rxnode.rate_consts_dict.update(fit_sci.get_fitted_rate_const_dict())
config = SolverConfig(**fit_sci.get_solver_config_args())

## 数値積分を実行する  

In [ ]:
# 最適化結果および与えたデータから、solve_ivp に渡す引数を作成
# 1. ODE システムと system_rhs を取得（builded_rxnode.rate_consts_dict は上記で更新済み）
ode_construct = builded_rxnode.get_ode_system()
(system_of_equations, sympy_symbol_dict, ode_system,
 function_names, rate_consts_dict) = ode_construct
system_rhs = create_system_rhs(ode_system, function_names)

# 2. 実験データの時間点を t_eval に使用（任意・省略可）
t_eval = np.array(df1.iloc[:, 0].values, dtype=float)

# 3. solve_ivp に渡す引数を構築（config を引き継ぎ、fun と t_eval を追加）
solve_ivp_kwargs = {
    'fun': system_rhs,
    **vars(config),
    't_eval': t_eval,
}

In [ ]:
# 基本的な数値積分 -インスタンス化-
solved_rxnode = RxnODEsolver(builded_rxnode, config)

In [ ]:
ode_construct, sol = solved_rxnode.solve_system()

In [ ]:
# シミュレーション曲線に実験データ点を重ねる（線と点で色を揃える）
solved_rxnode.solution_plot(expdata_df=[df1, df2, df3, df4], species=['oXy', 'mXy', 'pXy'], subplot_layout=(2,2))